# example_dq_rule_smoke_test — FabricOps DQ rule smoke test example

One time Microsoft Fabric validation notebook for proving each FabricOps DQ rule type against sample Spark data before teams trust the rules in production.

This notebook is intentionally stored under `templates/notebooks` with an `example_` prefix because it is a release-specific validation aid rather than a reusable production workflow template.

## 1. Run `00_env_config`

Load the shared FabricOps environment, configured metadata target, and Spark session bindings before seeding or enforcing rules.

In [ ]:
%run 00_env_config


## 2. Import public notebook APIs

Use public FabricOps notebook-friendly APIs for metadata writes and rule enforcement.

In [ ]:
import json

from pyspark.sql import functions as F

from fabricops_kit import run_table_guardrails, write_lakehouse_table
from fabricops_kit.config.shared import get_current_audit_timestamp
from fabricops_kit.governance_review import DQ_RULE_TYPES


## 3. Create sample Spark data

The sample rows intentionally mix valid and invalid values so every supported rule type has a known expected `failed_count`.

In [ ]:
SMOKE_DATASET_NAME = "fabricops_validation"
SMOKE_TABLE_NAME = "dq_rule_smoke_test"
SMOKE_ERROR_TABLE_NAME = "dq_rule_smoke_test_error_rollup"
SMOKE_PASS_TABLE_NAME = "dq_rule_smoke_test_pass_rollup"

sample_rows = [
    {
        "row_id": 1,
        "not_null_col": "ok",
        "null_rate_col": None,
        "non_empty_col": "filled",
        "unique_col": "A",
        "combo_a": 1,
        "combo_b": "A",
        "accepted_col": "red",
        "blocked_col": "ok",
        "between_col": 5,
        "gt_col": 11,
        "gte_col": 10,
        "lt_col": 4,
        "lte_col": 5,
        "regex_col": "ABC-123",
        "future_date_col": "2024-01-01",
        "date_between_col": "2024-01-15",
        "freshness_col": "2999-01-01",
        "max_age_col": "2999-01-01",
        "pair_a": 1,
        "pair_b": 1,
        "gte_a": 2,
        "gte_b": 1,
        "gt_a": 2,
        "gt_b": 1,
        "required_flag": "Y",
        "required_col": "present",
        "value_flag": "Y",
        "value_col": "yes",
        "expr_num_a": 2,
        "expr_num_b": 1,
        "pass_col": "present",
    },
    {
        "row_id": 2,
        "not_null_col": None,
        "null_rate_col": None,
        "non_empty_col": "",
        "unique_col": "B",
        "combo_a": 1,
        "combo_b": "A",
        "accepted_col": "blue",
        "blocked_col": "blocked",
        "between_col": 10,
        "gt_col": 10,
        "gte_col": 9,
        "lt_col": 5,
        "lte_col": 6,
        "regex_col": "bad",
        "future_date_col": "2999-01-01",
        "date_between_col": "2023-12-31",
        "freshness_col": "2000-01-01",
        "max_age_col": "2000-01-01",
        "pair_a": 2,
        "pair_b": 3,
        "gte_a": 1,
        "gte_b": 2,
        "gt_a": 1,
        "gt_b": 1,
        "required_flag": "Y",
        "required_col": None,
        "value_flag": "Y",
        "value_col": "no",
        "expr_num_a": 1,
        "expr_num_b": 2,
        "pass_col": "present",
    },
    {
        "row_id": 3,
        "not_null_col": "ok",
        "null_rate_col": "x",
        "non_empty_col": "   ",
        "unique_col": "B",
        "combo_a": 2,
        "combo_b": "B",
        "accepted_col": "green",
        "blocked_col": "ok",
        "between_col": 15,
        "gt_col": 12,
        "gte_col": 11,
        "lt_col": 3,
        "lte_col": 4,
        "regex_col": "XYZ-999",
        "future_date_col": "2024-06-01",
        "date_between_col": "2024-06-01",
        "freshness_col": "2999-01-01",
        "max_age_col": "2999-01-01",
        "pair_a": None,
        "pair_b": None,
        "gte_a": None,
        "gte_b": None,
        "gt_a": None,
        "gt_b": None,
        "required_flag": "N",
        "required_col": None,
        "value_flag": "N",
        "value_col": "no",
        "expr_num_a": 3,
        "expr_num_b": 3,
        "pass_col": "present",
    },
    {
        "row_id": 4,
        "not_null_col": "ok",
        "null_rate_col": "y",
        "non_empty_col": None,
        "unique_col": "C",
        "combo_a": 3,
        "combo_b": "C",
        "accepted_col": None,
        "blocked_col": None,
        "between_col": 25,
        "gt_col": None,
        "gte_col": None,
        "lt_col": None,
        "lte_col": None,
        "regex_col": None,
        "future_date_col": None,
        "date_between_col": None,
        "freshness_col": None,
        "max_age_col": None,
        "pair_a": 4,
        "pair_b": 4,
        "gte_a": 3,
        "gte_b": 3,
        "gt_a": 3,
        "gt_b": 2,
        "required_flag": "Y",
        "required_col": "",
        "value_flag": "Y",
        "value_col": None,
        "expr_num_a": None,
        "expr_num_b": 1,
        "pass_col": "present",
    },
    {
        "row_id": 5,
        "not_null_col": "ok",
        "null_rate_col": "z",
        "non_empty_col": "filled",
        "unique_col": "D",
        "combo_a": 4,
        "combo_b": "D",
        "accepted_col": "red",
        "blocked_col": "bad",
        "between_col": None,
        "gt_col": 9,
        "gte_col": 8,
        "lt_col": 6,
        "lte_col": 7,
        "regex_col": "ABC-12",
        "future_date_col": "2024-12-31",
        "date_between_col": "2025-01-01",
        "freshness_col": "2999-01-01",
        "max_age_col": "2999-01-01",
        "pair_a": 5,
        "pair_b": 6,
        "gte_a": None,
        "gte_b": 1,
        "gt_a": None,
        "gt_b": 1,
        "required_flag": "N",
        "required_col": None,
        "value_flag": "N",
        "value_col": "no",
        "expr_num_a": 5,
        "expr_num_b": 4,
        "pass_col": "present",
    },
]

sample_df = spark.createDataFrame(sample_rows)
display(sample_df)


## 4. Seed active governance-approved DQ rules into `METADATA_GUARDRAIL_RULES`

The smoke test uses stable dataset, table, and rule identifiers. Reruns append fresh reviewed metadata events scoped to the smoke test tables instead of touching normal project tables.

In [ ]:
# DQ_RULE_TYPES is imported only to detect validation coverage drift when FabricOps adds, removes,
# or reorders supported rule types. Enforcement in this smoke test still uses the public
# notebook API, run_table_guardrails.
SUPPORTED_RULE_TYPES = list(DQ_RULE_TYPES)

EXPECTED_FAILED_COUNTS = {
    "not_null": 1,
    "null_rate_below": 2,
    "non_empty_string": 3,
    "unique": 2,
    "unique_combination": 2,
    "accepted_values": 1,
    "not_in_values": 2,
    "between": 2,
    "greater_than": 2,
    "greater_than_or_equal": 2,
    "less_than": 2,
    "less_than_or_equal": 2,
    "regex_match": 2,
    "date_not_future": 1,
    "date_between": 2,
    "freshness": 1,
    "max_age_days": 1,
    "column_pair_equal": 2,
    "column_a_gte_column_b": 2,
    "column_a_gt_column_b": 2,
    "required_when": 2,
    "value_when": 2,
    "expression_true": 2,
}

RULE_DEFINITIONS = [
    {"rule_type": "not_null", "columns": ["not_null_col"]},
    {"rule_type": "null_rate_below", "columns": ["null_rate_col"], "max_null_percent": 20},
    {"rule_type": "non_empty_string", "columns": ["non_empty_col"]},
    {"rule_type": "unique", "columns": ["unique_col"]},
    {"rule_type": "unique_combination", "columns": ["combo_a", "combo_b"]},
    {"rule_type": "accepted_values", "columns": ["accepted_col"], "allowed_values": ["red", "blue"]},
    {"rule_type": "not_in_values", "columns": ["blocked_col"], "blocked_values": ["blocked", "bad"]},
    {"rule_type": "between", "columns": ["between_col"], "min_value": 10, "max_value": 20},
    {"rule_type": "greater_than", "columns": ["gt_col"], "value": 10},
    {"rule_type": "greater_than_or_equal", "columns": ["gte_col"], "value": 10},
    {"rule_type": "less_than", "columns": ["lt_col"], "value": 5},
    {"rule_type": "less_than_or_equal", "columns": ["lte_col"], "value": 5},
    {"rule_type": "regex_match", "columns": ["regex_col"], "regex_pattern": r"^[A-Z]{3}-\d{3}$"},
    {"rule_type": "date_not_future", "columns": ["future_date_col"]},
    {"rule_type": "date_between", "columns": ["date_between_col"], "min_value": "2024-01-01", "max_value": "2024-12-31"},
    {"rule_type": "freshness", "columns": ["freshness_col"], "max_age_days": 30},
    {"rule_type": "max_age_days", "columns": ["max_age_col"], "max_age_days": 30},
    {"rule_type": "column_pair_equal", "columns": ["pair_a", "pair_b"]},
    {"rule_type": "column_a_gte_column_b", "columns": ["gte_a", "gte_b"]},
    {"rule_type": "column_a_gt_column_b", "columns": ["gt_a", "gt_b"]},
    {"rule_type": "required_when", "columns": ["required_col"], "condition": "required_flag = 'Y'"},
    {"rule_type": "value_when", "columns": ["value_col"], "condition": "value_flag = 'Y'", "expected_value": "yes"},
    {"rule_type": "expression_true", "expression": "expr_num_a > expr_num_b"},
]

if [rule["rule_type"] for rule in RULE_DEFINITIONS] != SUPPORTED_RULE_TYPES:
    raise AssertionError("Smoke test rule definitions must cover every supported DQ rule type exactly once.")

if set(EXPECTED_FAILED_COUNTS) != set(SUPPORTED_RULE_TYPES):
    raise AssertionError("Expected failed-count mapping must cover every supported DQ rule type.")

now_utc = get_current_audit_timestamp(config=CONFIG, drop_microseconds=False)

def metadata_key(dataset_name, table_name, column_name=""):
    return "||".join([str(ENV), dataset_name, table_name, column_name])


def build_rule_rows(table_name, rule_definitions, *, severity="warning", expected_counts=None):
    rows = []
    for position, rule in enumerate(rule_definitions, start=1):
        rule_type = rule["rule_type"]
        rule_id = f"fabricops_dq_smoke__{table_name}__{rule_type}"
        columns = list(rule.get("columns", []))
        parameters = {key: value for key, value in rule.items() if key not in {"rule_type"}}
        description = f"Smoke test rule for {rule_type}"
        if expected_counts is not None:
            description = f"{description}; expected_failed_count={expected_counts[rule_type]}"
        first_column = columns[0] if columns else ""
        rows.append(
            {
                "rule_key": metadata_key(SMOKE_DATASET_NAME, table_name, rule_id),
                "rule_id": rule_id,
                "metadata_column_key": metadata_key(SMOKE_DATASET_NAME, table_name, first_column),
                "metadata_table_key": metadata_key(SMOKE_DATASET_NAME, table_name),
                "environment_name": ENV,
                "dataset_name": SMOKE_DATASET_NAME,
                "table_name": table_name,
                "column_name": first_column,
                "guardrail_type": "dq",
                "rule_type": rule_type,
                "rule_parameters_json": json.dumps(parameters, sort_keys=True),
                "severity": severity,
                "description": description,
                "is_active": True,
                "review_status": "governance_approved",
                "author_role": "smoke_test",
                "created_by": "fabricops_dq_smoke_test",
                "created_at": now_utc,
                "approved_by": "fabricops_dq_smoke_test",
                "approved_at": now_utc,
                "suggestion_json": "{}",
                "action_type": "created",
                "source_notebook_type": "example_dq_rule_smoke_test",
                "source_notebook_id": "",
                "source_workspace_id": "",
                "superseded_by_rule_key": "",
                "notes": "Public-safe DQ smoke test seed rule.",
                "_committed_at": now_utc,
                "_committed_by": "fabricops_dq_smoke_test",
                "_workspace_name": "",
                "_notebook_name": "dq_rule_smoke_test",
                "_metadata_lakehouse_name": CONFIG.path_config.paths[ENV]["metadata"].name,
                "_activity_id": "fabricops_dq_smoke_test",
            }
        )
    return rows

warning_rule_rows = build_rule_rows(SMOKE_TABLE_NAME, RULE_DEFINITIONS, severity="warning", expected_counts=EXPECTED_FAILED_COUNTS)
error_rule_rows = build_rule_rows(
    SMOKE_ERROR_TABLE_NAME,
    [{"rule_type": "not_null", "columns": ["not_null_col"]}],
    severity="error",
)
pass_rule_rows = build_rule_rows(
    SMOKE_PASS_TABLE_NAME,
    [{"rule_type": "not_null", "columns": ["pass_col"]}],
    severity="warning",
)

seeded_rule_df = spark.createDataFrame(warning_rule_rows + error_rule_rows + pass_rule_rows)
write_lakehouse_table(seeded_rule_df, "METADATA_GUARDRAIL_RULES", target="metadata", schema=METADATA_SCHEMA, mode="append")
display(seeded_rule_df.orderBy("table_name", "rule_id"))


## 5. Run DQ enforcement and display validation evidence

`run_table_guardrails` reads active governance-approved DQ rules from the configured metadata target as part of the supported pipeline guardrail path and evaluates them against the sample DataFrame.

In [ ]:
def run_smoke_guardrails(table_name):
    table_configs = [
        {
            "key": table_name,
            "df": sample_df,
            "dataset_name": SMOKE_DATASET_NAME,
            "table_name": table_name,
            "expected_schema": sample_df.schema,
            "freshness_column": "",
            "profile_mode": "skip",
            "dq_preset": "active_rules",
            "write_guardrail_results": False,
        }
    ]
    guardrail_bundle = run_table_guardrails(
        table_configs,
        context={"config": CONFIG, "env": ENV},
        run_id="fabricops_dq_smoke_test",
        spark_session=spark,
        mode="profile",
        stop_on_failure=False,
    )
    return guardrail_bundle["dq_results"][table_name]


warning_result = run_smoke_guardrails(SMOKE_TABLE_NAME)
warning_checks_df = spark.createDataFrame(warning_result["checks"])

print(warning_result["message"])
display(warning_checks_df.orderBy("rule_type"))
display(warning_result["dataframe"].select(*sample_df.columns, "_dq_check_status", "_dq_failed_rules").orderBy("row_id"))
display(spark.createDataFrame([warning_result["summary"]]))


## 6. Assert expected per-rule failures and roll-up behavior

The notebook raises `AssertionError` if any supported rule type, failed count, tagged DataFrame status, or guardrail roll-up behaves differently from the expected result.

In [ ]:
actual_counts = {
    row["rule_type"]: int(row["failed_count"])
    for row in warning_checks_df.select("rule_type", "failed_count").collect()
}

if actual_counts != EXPECTED_FAILED_COUNTS:
    raise AssertionError(f"Unexpected DQ failed counts. Expected {EXPECTED_FAILED_COUNTS}; got {actual_counts}")

if warning_result["status"] != "warning" or warning_result["can_continue"] is not True:
    raise AssertionError(f"Warning roll-up should be status='warning' and can_continue=True; got {warning_result}")

if not all(check["status"] == "warning" for check in warning_result["checks"]):
    raise AssertionError("Every smoke-test rule in the warning run should fail with warning severity.")

warning_summary = warning_result["summary"]
if warning_summary["DQ_STATUS"] != "warning":
    raise AssertionError(f"Warning summary should report DQ_STATUS='warning'; got {warning_summary}")

error_result = run_smoke_guardrails(SMOKE_ERROR_TABLE_NAME)
if error_result["status"] != "failed" or error_result["can_continue"] is not False:
    raise AssertionError(f"Error roll-up should be status='failed' and can_continue=False; got {error_result}")

pass_result = run_smoke_guardrails(SMOKE_PASS_TABLE_NAME)
if pass_result["status"] != "passed" or pass_result["can_continue"] is not True:
    raise AssertionError(f"Passing roll-up should be status='passed' and can_continue=True; got {pass_result}")

if not pass_result["checks"] or not all(check["passed"] and check["status"] == "passed" for check in pass_result["checks"]):
    raise AssertionError(f"Passing rules should produce passed checks; got {pass_result['checks']}")

tagged_statuses = {row["_dq_check_status"] for row in warning_result["dataframe"].select("_dq_check_status").distinct().collect()}
if "warning" not in tagged_statuses:
    raise AssertionError(f"Tagged DataFrame should include warning rows; got statuses {tagged_statuses}")

print("DQ rule smoke test passed for all supported rule types.")
print("Warning roll-up:", {"status": warning_result["status"], "can_continue": warning_result["can_continue"]})
print("Error roll-up:", {"status": error_result["status"], "can_continue": error_result["can_continue"]})
print("Passing roll-up:", {"status": pass_result["status"], "can_continue": pass_result["can_continue"]})
